<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-07-mcp-and-cloud-run/lesson-7.3-agent-mcp/practice/GCP_Capstone_7.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 7.3 — Connect Agent to Remote MCP

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup

Install the ADK + MCP stack, authenticate with Application Default Credentials, and point the SDK at Vertex AI. Run this cell first — every exercise below depends on the client, imports, and env vars it sets up.

In [ ]:
%%bash
pip install -q google-adk google-genai google-auth fastmcp

In [ ]:
# Application Default Credentials — never API keys
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE to your project
LOCATION = 'global'                # Gemini 3.x generation is served from the global Vertex endpoint (per-region comparison no longer applies)

import os
os.environ['GOOGLE_CLOUD_PROJECT'] = PROJECT_ID
os.environ['GOOGLE_CLOUD_LOCATION'] = LOCATION
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'

print(f'Project: {PROJECT_ID}  |  Location: {LOCATION}')

## Exercise 1: Connect to Local MCP
**Difficulty:** Easy

McpToolset with StreamableHTTPConnectionParams to localhost:8000/mcp.

1. Start your Lesson 7.1 server
2. Create McpToolset with local URL
3. Verify connection (tools listed when agent starts)

**Run this in a separate terminal / Cloud Shell** — it starts a long-running server that would block the notebook kernel:

```bash
# Start your Lesson 7.1 FastMCP server; it serves Streamable HTTP at http://localhost:8000/mcp
python documind_server.py
```

In [ ]:
from google.adk.tools.mcp_tool import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StreamableHTTPConnectionParams

# Step 2: connect to the local server (no auth needed on localhost)
local_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url='http://localhost:8000/mcp',
    ),
)

# Step 3: the toolset connects and discovers tools lazily when an agent starts.
print('McpToolset created (connects + lists tools when agent starts)')

## Exercise 2: Connect via Proxy
**Difficulty:** Easy

gcloud run services proxy. Connect McpToolset to localhost:3000/mcp.

1. Start Cloud Run proxy
2. McpToolset with localhost:3000/mcp (no auth)
3. Verify tools discovered

**Run this in a separate terminal / Cloud Shell** — the proxy runs in the foreground and would block the notebook kernel:

```bash
# Proxy the deployed Cloud Run service to a local port; the proxy injects your identity token automatically.
gcloud run services proxy documind-mcp-server --port=3000 --region=us-central1
```

In [ ]:
# Step 2: point McpToolset at the proxied port — the proxy authenticates for you.
proxy_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url='http://localhost:3000/mcp',
    ),
)

# Step 3: tools are discovered through the proxy when the agent starts.
print('McpToolset via Cloud Run proxy created (proxy auto-authenticates)')

## Exercise 3: Agent Calls MCP Tool
**Difficulty:** Easy

LlmAgent with McpToolset. Send query. Verify tool called and answer returned.

1. LlmAgent(tools=[mcp_toolset])
2. Send "Find financial documents"
3. Verify search_documents called via MCP

In [ ]:
from google.adk.agents import LlmAgent

# Step 1: wrap the MCP toolset in an agent
agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_agent',
    instruction='You are DocuMind AI. Use tools to search documents, '
                'calculate costs, and classify content.',
    tools=[local_tools],
)
print(f'Agent created: {agent.name}')

In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

# Step 2 + 3: send a query and watch the MCP tool get called.
# NOTE: requires the Lesson 7.1 server running locally (Exercise 1).
async def test_agent(query, use_agent=agent):
    session_service = InMemorySessionService()
    runner = Runner(
        agent=use_agent,
        app_name='documind',
        session_service=session_service,
    )
    session = await session_service.create_session(
        app_name='documind', user_id='student')

    content = types.Content(
        role='user',
        parts=[types.Part.from_text(text=query)])

    print(f'Sending query: {query!r}')
    async for event in runner.run_async(
            user_id='student', session_id=session.id, new_message=content):
        if event.content and event.content.parts:
            for part in event.content.parts:
                if part.function_call:
                    print(f'Tool call via MCP: {part.function_call.name}')
                if part.text:
                    print(f'Agent: {part.text}')

# await test_agent('Find financial documents')

## Exercise 4: IAM Token Auth
**Difficulty:** Medium

Generate ID token. Pass in headers. Connect to Cloud Run MCP server.

1. google.oauth2.id_token.fetch_id_token()
2. headers={'Authorization': f'Bearer {token}'}
3. Verify authenticated connection works

In [ ]:
import google.auth.transport.requests
import google.oauth2.id_token

# Replace with your deployed Cloud Run URL
CLOUD_RUN_URL = 'https://documind-mcp-server-HASH-uc.a.run.app'
MCP_URL = f'{CLOUD_RUN_URL}/mcp'

# Step 1: fetch an OIDC identity token whose audience is the service base URL
def get_id_token(url):
    audience = url.split('/mcp')[0]
    request = google.auth.transport.requests.Request()
    return google.oauth2.id_token.fetch_id_token(request, audience)

try:
    token = get_id_token(MCP_URL)
    print(f'Token generated: {token[:30]}...')
except Exception as e:
    print(f'Token error (expected if no Cloud Run deployed): {e}')
    token = 'dummy-for-local-testing'

In [ ]:
# Step 2: pass the token as a Bearer header on the MCP connection
auth_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=MCP_URL,
        headers={'Authorization': f'Bearer {token}'},
    ),
)

# Step 3: an agent built on the authenticated toolset can now reach the private service
agent_cloud = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_cloud_agent',
    instruction='You are DocuMind AI connected to Cloud Run tools.',
    tools=[auth_tools],
)
print('Cloud agent created with authenticated (IAM) MCP connection')

## Exercise 5: BigQuery via Toolbox
**Difficulty:** Medium

McpToolset connected to BQ Toolbox. Call query-document-analytics.

1. Deploy Toolbox with BQ tools.yaml
2. McpToolset with tool_filter
3. Agent calls BQ tool through MCP

In [ ]:
# Step 1 + 2: connect to the Toolbox service deployed in Lesson 7.2 and
# expose only the two BigQuery tools we want via tool_filter.
TOOLBOX_URL = 'https://documind-toolbox-HASH-uc.a.run.app'

try:
    _tok = get_id_token(TOOLBOX_URL)
except Exception:
    _tok = 'PLACEHOLDER-TOKEN'  # deploy the Lesson 7.2 service, set the URL above, and run: gcloud auth application-default login

bq_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f'{TOOLBOX_URL}/mcp',
        headers={'Authorization': f'Bearer {_tok}'},
    ),
    tool_filter=['query-document-analytics', 'predict-document-category'],
)
print('BigQuery MCP Toolset configured')

# BQML tools.yaml example for the predict-document-category tool:
print('''
tools.yaml snippet for BQML prediction:
---
kind: tool
name: predict-document-category
type: bigquery-sql
source: documind-bq
description: Predict document category using BQML model.
parameters:
  - name: document_text
    type: string
statement: >
  SELECT * FROM ML.PREDICT(
    MODEL `documind.document_classifier`,
    (SELECT @document_text AS text_content));
''')

In [ ]:
# Step 3: an agent that queries analytics through the BigQuery Toolbox over MCP
bq_agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_bq_agent',
    instruction='You are DocuMind AI. Use the BigQuery tools to answer '
                'analytics questions about processed documents.',
    tools=[bq_tools],
)
print(f'BQ agent created: {bq_agent.name}')
# await test_agent('How many documents were processed this month?', use_agent=bq_agent)

## Exercise 6: tool_filter
**Difficulty:** Medium

Server has 10 tools. Use tool_filter to expose only 3. Verify agent sees only 3.

1. tool_filter=['tool_a', 'tool_b', 'tool_c']
2. Or: tool_filter=lambda t: t.name.startswith('doc_')
3. Verify filtered tool count

In [ ]:
# Step 1: allow-list form — expose only the three named tools
filtered_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url='http://localhost:8000/mcp',
    ),
    tool_filter=['search_documents', 'calculate_cost', 'classify_document'],
)

# Step 2: predicate form — expose only tools whose name starts with 'doc_'
def doc_only(tool, ctx=None):
    return tool.name.startswith('doc_')

prefix_filtered_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url='http://localhost:8000/mcp',
    ),
    tool_filter=doc_only,
)

print('Allow-list toolset and predicate toolset configured')

In [ ]:
# Step 3: verify the agent only sees the whitelisted tools.
# get_tools() returns the post-filter tool objects; requires the server running.
async def count_tools(toolset):
    tools = await toolset.get_tools()
    names = [t.name for t in tools]
    print(f'{len(names)} tools exposed: {names}')
    return names

# await count_tools(filtered_tools)   # expect exactly 3 names

## Exercise 7: Multi-Server Agent
**Difficulty:** Challenge

Agent with 2+ McpToolsets. Send queries that route to different servers.

1. McpToolset for FastMCP + McpToolset for Toolbox
2. "Find docs" -> Server 1. "Query analytics" -> Server 2.
3. Verify Gemini routes correctly

In [ ]:
# Step 1: give one agent several tool sources — two MCP servers plus a plain function.
def format_report(data: dict, template: str = 'default') -> str:
    """Format extracted data into a structured report."""
    return f'Report ({template}): {data}'

multi_agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_multi_agent',
    instruction="""You are DocuMind AI with access to:
- Document search, cost, and classification tools (FastMCP server)
- BigQuery analytics and ML prediction tools (Toolbox)
- Report formatting
Use the right tool for each query.""",
    tools=[
        local_tools,     # FastMCP server (search/cost/classify)
        bq_tools,        # BigQuery Toolbox (analytics/prediction)
        format_report,   # direct function tool
    ],
)
print(f'Multi-server agent: {multi_agent.name}')
print(f'Tool sources: {len(multi_agent.tools)}')

In [ ]:
# Step 2 + 3: Gemini inspects each query and routes to the correct server.
# 'Find docs'        -> search_documents on the FastMCP server
# 'Query analytics'  -> query-document-analytics on the Toolbox server
# await test_agent('Find financial documents from Q3', use_agent=multi_agent)
# await test_agent('Query analytics: total documents processed by category', use_agent=multi_agent)
print('Send both queries and confirm each triggers a tool on a different server.')

## Exercise 8: Document AI MCP Server
**Difficulty:** Challenge

Build FastMCP server wrapping Document AI OCR. Deploy. Connect from agent.

1. @mcp.tool wrapping documentai client
2. Deploy to Cloud Run
3. McpToolset in agent with IAM auth

In [ ]:
# Step 1: a FastMCP server exposing a single extract_text tool backed by Document AI OCR.
# Save this as docai_server.py, then add requirements.txt (fastmcp, google-cloud-documentai).
docai_server = '''
import os
from fastmcp import FastMCP
from google.cloud import documentai

mcp = FastMCP("docai-ocr")

PROJECT_ID = os.environ["GOOGLE_CLOUD_PROJECT"]
LOCATION = os.environ.get("DOCAI_LOCATION", "us")
PROCESSOR_ID = os.environ["DOCAI_PROCESSOR_ID"]  # OCR processor

_client = documentai.DocumentProcessorServiceClient()
_name = _client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

@mcp.tool
def extract_text(content_base64: str, mime_type: str = "application/pdf") -> str:
    """Run Document AI OCR on a base64-encoded document and return the text."""
    import base64
    raw = base64.b64decode(content_base64)
    raw_doc = documentai.RawDocument(content=raw, mime_type=mime_type)
    request = documentai.ProcessRequest(name=_name, raw_document=raw_doc)
    result = _client.process_document(request=request)
    return result.document.text

if __name__ == "__main__":
    # Streamable HTTP transport for Cloud Run
    mcp.run(transport="http", host="0.0.0.0", port=int(os.environ.get("PORT", 8080)))
'''

with open('docai_server.py', 'w') as f:
    f.write(docai_server)
print('Wrote docai_server.py')

In [ ]:
%%bash
# Step 2: deploy the Document AI MCP server to Cloud Run, private (IAM-only, no public access).
# The source-based deploy needs a requirements.txt next to docai_server.py for the buildpack.
cat > requirements.txt <<'EOF'
fastmcp>=2.0
google-cloud-documentai>=2.0
EOF
gcloud run deploy documind-docai-mcp \
  --source . \
  --region=us-central1 \
  --no-allow-unauthenticated \
  --set-env-vars=DOCAI_LOCATION=us,DOCAI_PROCESSOR_ID=YOUR_OCR_PROCESSOR_ID

In [ ]:
# Step 3: connect an agent to the private Document AI MCP server using an IAM identity token.
DOCAI_URL = 'https://documind-docai-mcp-HASH-uc.a.run.app'

try:
    _tok = get_id_token(DOCAI_URL)
except Exception:
    _tok = 'PLACEHOLDER-TOKEN'  # deploy the Lesson 7.2 service, set the URL above, and run: gcloud auth application-default login

docai_tools = McpToolset(
    connection_params=StreamableHTTPConnectionParams(
        url=f'{DOCAI_URL}/mcp',
        headers={'Authorization': f'Bearer {_tok}'},
    ),
)

docai_agent = LlmAgent(
    model='gemini-3.6-flash',
    name='documind_docai_agent',
    instruction='You are DocuMind AI. Use extract_text to OCR documents '
                'via the Document AI MCP server.',
    tools=[docai_tools],
)
print(f'Document AI agent created: {docai_agent.name}')
# await test_agent('Extract the text from this scanned invoice', use_agent=docai_agent)